# 04 — Classical vs Quantum ML: Full Comparison

Loads results from Phases 2 and 3, generates all publication-quality static figures
and interactive Plotly dashboards. Produces the final `results/` outputs.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from pathlib import Path
import geopandas as gpd
import yaml

from src.features.engineering import load_config, load_splits
from src.models.classical import ClassicalModelTrainer
from src.models.quantum import QuantumModelTrainer
from src.evaluation.metrics import evaluate_model, compare_models, print_comparison_table
from src.visualization import static_plots as sp
from src.visualization import interactive as iv

config = load_config('../config/config.yaml')
CLASS_NAMES = config['labeling']['class_names']
FIGURES_DIR = Path('../results/figures')
MODELS_DIR = Path('../results/models')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete')

## 1. Load All Results

In [ ]:
# Load data splits
splits, quantum = load_splits(config)
X_test  = splits['X_test']
y_test  = splits['y_test']
X_test_q = quantum['X_test_q']
y_test_q = quantum['y_test_q']

# Load classical comparison CSV
classical_comp = pd.read_csv('../results/classical_comparison.csv')
print('Classical results:')
print(classical_comp[['model', 'accuracy', 'f1_macro', 'training_time_s']].to_string(index=False))

In [ ]:
# Load quantum results
with open('../results/quantum_results.json') as f:
    quantum_json = json.load(f)

print('Quantum results:')
for name, r in quantum_json.items():
    print(f"  {name}: acc={r['accuracy']:.4f}, f1={r['f1_macro']:.4f}, "
          f"time={r['training_time']:.1f}s")

In [ ]:
# Re-run evaluation to get confusion matrices (needed for figures)
# Reload classical models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trainer_c = ClassicalModelTrainer(config)

model_keys = {
    'Random Forest': 'random_forest',
    'SVM (RBF)': 'svm',
    'XGBoost': 'xgboost',
}

all_results = {}

for display_name, key in model_keys.items():
    pkl_path = MODELS_DIR / f'{key}.pkl'
    if pkl_path.exists():
        model = joblib.load(pkl_path)
        metrics = evaluate_model(
            predict_fn=lambda X, m=model: m.predict(X),
            X_test=X_test, y_test=y_test,
            model_name=display_name, class_names=CLASS_NAMES,
        )
        # Get training time from saved CSV
        row = classical_comp[classical_comp['model'] == display_name]
        metrics['training_time'] = float(row['training_time_s'].values[0]) if len(row) else 0.0
        all_results[display_name] = metrics
    else:
        print(f'  [MISSING] {pkl_path} — re-run notebook 02 first')

# Neural Network
nn_meta_path = MODELS_DIR / 'neural_network_meta.pkl'
nn_state_path = MODELS_DIR / 'neural_network_state_dict.pt'
if nn_state_path.exists() and nn_meta_path.exists():
    from src.models.classical import ClimateNN
    from src.features.engineering import CLASSICAL_FEATURES
    nn_meta = joblib.load(nn_meta_path)
    n_features = len([f for f in CLASSICAL_FEATURES if f in pd.read_csv('../data/processed/merra2_india_labeled.csv').columns])
    nn_model = ClimateNN(
        input_dim=X_test.shape[1],
        hidden_dims=config['classical_ml']['neural_net']['hidden_dims'],
        n_classes=len(CLASS_NAMES),
        dropout=config['classical_ml']['neural_net']['dropout'],
    ).to(device)
    nn_model.load_state_dict(torch.load(nn_state_path, map_location=device))
    nn_model.eval()

    def nn_predict(X):
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            return nn_model(X_t).argmax(dim=1).cpu().numpy()

    nn_metrics = evaluate_model(
        predict_fn=nn_predict, X_test=X_test, y_test=y_test,
        model_name='Neural Network', class_names=CLASS_NAMES,
    )
    row = classical_comp[classical_comp['model'] == 'Neural Network']
    nn_metrics['training_time'] = float(row['training_time_s'].values[0]) if len(row) else 0.0
    nn_metrics['train_losses'] = nn_meta.get('train_losses', [])
    nn_metrics['val_losses'] = nn_meta.get('val_losses', [])
    all_results['Neural Network'] = nn_metrics

# Quantum models
trainer_q = QuantumModelTrainer(config)
for q_name in ['QSVC', 'VQC']:
    meta_path = MODELS_DIR / f'{q_name.lower()}_meta.pkl'
    pkl_path = MODELS_DIR / f'{q_name.lower()}.pkl'
    if pkl_path.exists():
        q_model = joblib.load(pkl_path)
        q_metrics = evaluate_model(
            predict_fn=lambda X, m=q_model: m.predict(X),
            X_test=X_test_q, y_test=y_test_q,
            model_name=q_name, class_names=CLASS_NAMES,
        )
        q_data = quantum_json.get(q_name, {})
        q_metrics['training_time'] = q_data.get('training_time', 0.0)
        if meta_path.exists():
            meta = joblib.load(meta_path)
            q_metrics.update({k: v for k, v in meta.items()
                              if k not in q_metrics})
        all_results[q_name] = q_metrics
    else:
        print(f'  [MISSING] {pkl_path} — re-run notebook 03 first')

print(f"\nLoaded {len(all_results)} models: {list(all_results.keys())}")

## 2. Full Comparison Table

In [ ]:
comp_df = compare_models(all_results)
comp_df.to_csv('../results/full_comparison.csv', index=False)
print_comparison_table(comp_df)
comp_df

## 3. Static Publication Figures

In [ ]:
# Figure 1: Model comparison bar chart
sp.plot_model_comparison_bar(
    comp_df, FIGURES_DIR / 'model_comparison_bar.png'
)
print('Saved: model_comparison_bar.png')

In [ ]:
# Figure 2: All confusion matrices
sp.plot_confusion_matrices(
    all_results, FIGURES_DIR / 'all_confusion_matrices.png',
    class_names=CLASS_NAMES,
)
print('Saved: all_confusion_matrices.png')

In [ ]:
# Figure 3: Scalability
classical_scaling = pd.read_csv('../results/classical_scaling.csv') \
    if Path('../results/classical_scaling.csv').exists() else pd.DataFrame()
quantum_scaling = pd.read_csv('../results/quantum_scaling.csv') \
    if Path('../results/quantum_scaling.csv').exists() else pd.DataFrame()

if not classical_scaling.empty and not quantum_scaling.empty:
    sp.plot_scalability(
        classical_scaling, quantum_scaling,
        FIGURES_DIR / 'scalability_comparison.png',
    )
    print('Saved: scalability_comparison.png')

In [ ]:
# Figure 4: Learning curves
nn_r = all_results.get('Neural Network', {})
vqc_r = all_results.get('VQC', {})
if nn_r.get('train_losses') and vqc_r.get('loss_history'):
    sp.plot_learning_curves(
        nn_r['train_losses'], nn_r['val_losses'],
        vqc_r['loss_history'],
        FIGURES_DIR / 'learning_curves.png',
    )
    print('Saved: learning_curves.png')

In [ ]:
# Figure 5: Feature importance
feature_names = splits.get('feature_names', [])
rf_imp = None
xgb_imp = None

rf_pkl = MODELS_DIR / 'random_forest.pkl'
xgb_pkl = MODELS_DIR / 'xgboost.pkl'
if rf_pkl.exists():
    rf_imp = joblib.load(rf_pkl).feature_importances_
if xgb_pkl.exists():
    xgb_imp = joblib.load(xgb_pkl).feature_importances_

if rf_imp is not None or xgb_imp is not None:
    sp.plot_feature_importance(
        rf_imp, xgb_imp, feature_names,
        FIGURES_DIR / 'feature_importance.png',
    )
    print('Saved: feature_importance.png')

In [ ]:
# Figure 6: Hero summary figure
sp.create_summary_figure(
    comp_df, all_results,
    vqc_loss=vqc_r.get('loss_history', []),
    save_path=FIGURES_DIR / 'summary_hero.png',
    class_names=CLASS_NAMES,
)
print('Saved: summary_hero.png (README hero image)')

## 4. Interactive Plotly Dashboard

In [ ]:
# Load India GeoJSON for choropleth
import geopandas as gpd
shapefile_path = Path('../data/shapefiles/india_states.geojson')
if shapefile_path.exists():
    gdf = gpd.read_file(shapefile_path)
    name_col = [c for c in gdf.columns
                if any(k in c.lower() for k in ['state','name','st_nm'])][0]
    gdf = gdf.rename(columns={name_col: 'state'})
    import json
    geojson = json.loads(gdf.to_json())
    print('GeoJSON loaded')
else:
    geojson = None
    print('[WARN] Shapefile not found — map figures will be skipped')

In [ ]:
# Model comparison dashboard
fig_dashboard = iv.model_comparison_dashboard(all_results)
fig_dashboard.show()

In [ ]:
# Quantum vs classical scatter
fig_scatter = iv.quantum_vs_classical_scatter(all_results)
fig_scatter.show()

In [ ]:
# 3D PCA scatter
fig_pca = iv.pca_3d_scatter(
    quantum['X_train_q'], quantum['y_train_q'],
    class_names=CLASS_NAMES,
)
fig_pca.show()

In [ ]:
# Quantum circuit info table
circuit_info = {
    'qsvc': quantum_json.get('QSVC', {}),
    'vqc': quantum_json.get('VQC', {}),
}
fig_circuit_table = iv.quantum_circuit_stats_table(circuit_info)
fig_circuit_table.show()

In [ ]:
# Climate data explorer (if shapefile available)
df_labeled = pd.read_csv('../data/processed/merra2_india_labeled.csv')
if geojson:
    fig_map = iv.india_climate_explorer(df_labeled, geojson, variable='T2M')
    fig_map.show()

In [ ]:
# Climate condition time series
fig_ts = iv.climate_condition_time_series(
    df_labeled,
    states=['Rajasthan', 'Kerala', 'Delhi'],
    class_names=CLASS_NAMES,
)
fig_ts.show()

In [ ]:
# Save full HTML dashboard
figures_to_export = [
    (fig_dashboard, 'Model Comparison Dashboard'),
    (fig_scatter, 'Training Time vs F1 Macro'),
    (fig_pca, 'PCA Feature Space (Quantum Training Data)'),
    (fig_circuit_table, 'Quantum Circuit Properties'),
    (fig_ts, 'Climate Condition Time Series'),
]
if geojson:
    figures_to_export.append((fig_map, 'India Climate Explorer'))

figs, titles = zip(*figures_to_export)
iv.save_html_dashboard(
    list(figs), list(titles),
    output_path=Path('../results/interactive_dashboard.html'),
)
print('Saved: results/interactive_dashboard.html')

## 5. Key Findings Summary

In [ ]:
best_classical = comp_df[~comp_df['model'].isin(['QSVC','VQC'])].iloc[0]
best_quantum   = comp_df[comp_df['model'].isin(['QSVC','VQC'])].iloc[0]
overall_best   = comp_df.iloc[0]

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print(f"\nBest Classical Model : {best_classical['model']}")
print(f"  Accuracy  : {best_classical['accuracy']:.4f}")
print(f"  F1 Macro  : {best_classical['f1_macro']:.4f}")
print(f"  Train Time: {best_classical['training_time_s']:.1f}s")

print(f"\nBest Quantum Model   : {best_quantum['model']}")
print(f"  Accuracy  : {best_quantum['accuracy']:.4f}")
print(f"  F1 Macro  : {best_quantum['f1_macro']:.4f}")
print(f"  Train Time: {best_quantum['training_time_s']:.1f}s")

gap_f1 = best_classical['f1_macro'] - best_quantum['f1_macro']
gap_time = best_quantum['training_time_s'] / max(best_classical['training_time_s'], 0.01)

print(f"\nPerformance gap (Classical - Quantum F1): {gap_f1:+.4f}")
print(f"Training time ratio (Quantum / Classical): {gap_time:.1f}x")

print("\nConclusion:")
if gap_f1 > 0.05:
    print("  Classical ML outperforms Quantum ML significantly.")
    print("  QML is constrained by the small training subset (400 samples)")
    print("  and limited qubit count (4 qubits on simulation).")
elif gap_f1 < -0.05:
    print("  Quantum ML outperforms Classical ML — notable result!")
else:
    print("  Classical and Quantum ML achieve comparable performance.")
    print("  Classical has a significant training time advantage.")

print("\nKey QML limitations observed:")
print("  - Training set restricted to 400 samples (simulation cost)")
print("  - 4 qubits => 4 PCA features only (information loss)")
print("  - Exponential simulation cost limits qubit scaling")
print("  - VQC convergence sensitive to optimizer choice")

## 6. Why Quantum ML May (or May Not) Outperform Classical ML

### Current State (NISQ era)

**Why QML likely underperforms here:**
1. **Training size**: Classical models use 9,000+ samples; QML uses 400 (due to O(n²) kernel matrix cost for QSVC and simulation overhead for VQC)
2. **Feature space**: PCA to 4 dimensions discards information; classical models use 14+ features
3. **Simulation overhead**: Statevector simulation on CPU doesn't reflect quantum hardware speedup
4. **Noise**: Real quantum hardware introduces gate errors that degrade performance

**Where QML might gain advantage:**
1. **Quantum kernel advantage**: If the data has structure that maps naturally to quantum Hilbert space (exponentially large feature space), QSVC can in theory separate classes that classical kernels cannot
2. **Future hardware**: Fault-tolerant quantum computers could run deep circuits beyond NISQ capabilities
3. **Quantum data**: If the input data itself is quantum (e.g., sensor readings from quantum devices), QML avoids costly classical encoding

### Research Directions
- **Quantum advantage for ML**: Still an open research question — no proven exponential advantage for classical ML tasks
- **Better encodings**: Amplitude encoding (vs angle encoding) can represent exponentially more features per qubit
- **Hybrid approaches**: Quantum kernels + classical SVMs, or variational circuits as layers in classical NNs